In [1]:
import pandas as pd


In [2]:
#데이터 로딩: 주문 상세
oi = pd.read_csv('../data/order_items.csv')
oi.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  500000 non-null  int64  
 1   order_id       500000 non-null  int64  
 2   product_id     500000 non-null  int64  
 3   quantity       500000 non-null  int64  
 4   unit_price     484887 non-null  float64
 5   discount       500000 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 22.9 MB


In [4]:
oi.describe()[['quantity', 'unit_price', 'discount']]

,quantity,unit_price,discount
count,500000.000000,484887.000000,500000.000000
mean,2.997008,50925.557707,0.249688
std,1.418193,48908.412198,0.144496
min,-2.000000,1000.000000,0.000000
25%,2.000000,15100.000000,0.120000
50%,3.000000,27000.000000,0.250000
75%,4.000000,73700.000000,0.370000
max,5.000000,291000.000000,0.500000


In [12]:
# 1. 진단: 중복, 이상값
before_rows = len(oi)
print(f'정제 전 행수 : {before_rows}')

# oi.duplicated() : boolean (True, False)
dup_cnt = oi.duplicated().sum()
print(f'완전 중복 행수 : {dup_cnt}')

neg_qty = (oi['quantity'] <= 0).sum()
print(f'수량 0 이하(이상값) 개수 : {neg_qty}')
print(f'수량 최솟값/최댓값 : {oi["quantity"].min()}, {oi["quantity"].max()}')

정제 전 행수 : 500000
완전 중복 행수 : 120
수량 0 이하(이상값) 개수 : 500
수량 최솟값/최댓값 : -2, 5


In [17]:
# 2. 규칙 적용: 중복제거후 수량 1 이상만 남긴다
oi_clean = oi.drop_duplicates()  #중복 삭젝
print(f'중복 제거 후 길이 : {len(oi_clean)}')
oi_clean = oi.clean[oi_clean['quantity'] > 0]
print(f'정제 후 행수 : {len(oi_clean)}')

중복 제거 후 길이 : 499880
정제 후 행수 : 499380


C:\Users\mega\AppData\Local\Temp\ipykernel_4108\2768736475.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  oi_clean = oi.clean[oi_clean['quantity'] > 0]


In [19]:
oi_clean = oi.drop_duplicates()
print(f"중복 제거 후 길이 : {len(oi_clean)}")

oi_clean = oi_clean[oi_clean["quantity"] > 0].copy()
print(f"정제 후 행수 : {len(oi_clean)}")

중복 제거 후 길이 : 499880
정제 후 행수 : 499380


In [20]:
import pandas as pd
prod = pd.read_csv('../data/products.csv')

In [21]:
prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    int64  
 1   product_name  500 non-null    str    
 2   category      500 non-null    str    
 3   price         500 non-null    str    
 4   cost          500 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.5 KB


In [ ]:
prod['price'].dtype #string

<StringDtype(na_value=nan)>

In [24]:
#1. 데이터 타입 변환: 숫자 변환 실패(문자열 오염) 개수 진단:
# fail_cnt = pd.to_numeric(prod['price'])  #value error -> "14,200"


In [29]:
# 변환할 수 있는 넘은 변환하고 에러나는 놈은 다른 방식으로 처리
# errors => raise, coerce, ignore
# raise(디폴트): 변환 실패시 예외 발생 -> 프로그램 중단
# coerce : 변환 실패 값을 NaN으로 대체 -> 프로그램 중단 X
# ignore : 원본 그대로 반환
fail_cnt =  pd.to_numeric(prod['price'], errors ="coerce").isna().sum() #46개

In [ ]:
#변환 실패한 값을 확인 -> 무엇이 문제인지 진단

prod.loc[
    pd.to_numeric(prod['price'], errors ="coerce").isna(), #행 
    'price'  #열
].head(10)

11     14,200
21     95,500
26      3,200
43     12300원
45     20,300
48    147,600
55     32,600
74     32500원
88     70,400
90    115800원
Name: price, dtype: str

In [32]:
# 규칙: '원', ',' 제거 후 숫자화
price_str = (prod['price'].astype(str)
            .str.replace('원', '', regex=False)
            .str.replace(',','', regex=False))
prod['price'] = pd.to_numeric(price_str, errors = 'coerce')
prod['price'].isna().sum()

np.int64(0)

In [34]:
# 음수 가격 확인
(prod['price'] < 0).sum()

prod.loc[
    prod['price'] < 0
]

,product_id,product_name,category,price,cost
87,217,데일리 잡지,도서,-1000.0,7700.0
181,352,데일리 시집,도서,-500.0,10600.0
310,268,스탠다드 노트북,전자,-1000.0,60500.0


In [35]:
#clip
oi = pd.read_csv('../data/order_items.csv')
print(f'최솟값: {oi["discount"].min()}, 최대값: {oi["discount"].max()}')

최솟값: 0.0, 최대값: 0.5
